In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages.utils import trim_messages
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os
load_dotenv()

C:\Users\bisho\AppData\Local\Temp\ipykernel_19972\357952205.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_tokens=512,
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [4]:
model.invoke("hello my name is bishoy")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': "Hello Bishoy! It's nice to meet you. How can I help you today?", 'extras': {'signature': 'El4KXAERTTIPtRgvgMWuYRmIP56OTk6+PucKblyhIdibUcaoN7TiH2aBp95P2TJwz3YdSK7Aq96S0Sxtt3YA1MW7egVsERl8ZRPPnvJ8LCCFP/vfnY2m6Jt3HNcPK5b0'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a07c98-6bdc-7123-a382-e621d869bd73-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 19, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}})

In [5]:
model.invoke("what is my name ?")

AIMessage(content=[{'type': 'text', 'text': "I don't know your name! You haven't told me yet. What is your name?", 'extras': {'signature': 'El4KXAERTTIPHcAor62Ia6uTbYwnQsyeowg/xd9sG4yqU7M4WAiyyXErGeF8ZaHTgJSwHP9AYnpaIya2nb293lfTARWKaQdeMdIVY1JikQKjRuWuZeOfJXAmGGG34gjf'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a07c98-6f5e-72b2-bfdb-a41ce146d425-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 21, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}})

## 1) Buffer Memory

### Give All Previous Chat Each Request You Do

In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

In [7]:
chain = prompt | model | StrOutputParser()

In [8]:
history = InMemoryChatMessageHistory()

In [9]:
conversation = RunnableWithMessageHistory(
    chain,
    lambda session_id: history,
    history_messages_key="history",
    input_messages_key="input",
)

c:\Users\bisho\miniconda3\envs\agent\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
conversation.invoke(
    {"input": "My name is Bishoy"},
    config={"configurable": {"session_id": "1"}},
)

'Nice to meet you, Bishoy! How can I help you today?'

In [11]:
conversation.invoke(
    {"input": "what's my name ?"},
    config={"configurable": {"session_id": "1"}}
)

'Your name is Bishoy!'

In [12]:
history.messages

[HumanMessage(content='My name is Bishoy', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you, Bishoy! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="what's my name ?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Your name is Bishoy!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 2) Buffer Window Memory

In [13]:
history = InMemoryChatMessageHistory()

In [14]:
history.messages

[]

In [15]:
history.add_user_message("My favorite language is Python.")
history.add_ai_message("Python is a great language.")

history.add_user_message("I am learning LangChain.")
history.add_ai_message("LangChain is useful for building LLM applications.")

history.add_user_message("I also want to learn CrewAI.")
history.add_ai_message("CrewAI is useful for building multi-agent systems.")

In [16]:
history.messages

[HumanMessage(content='My favorite language is Python.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Python is a great language.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I am learning LangChain.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='LangChain is useful for building LLM applications.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I also want to learn CrewAI.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='CrewAI is useful for building multi-agent systems.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [17]:
recent_history = history.messages[-4:]

In [18]:
recent_history

[HumanMessage(content='I am learning LangChain.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='LangChain is useful for building LLM applications.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I also want to learn CrewAI.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='CrewAI is useful for building multi-agent systems.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [19]:
response = chain.invoke({
    "history": recent_history,
    "input": "What am I currently learning?"
})

In [20]:
response

"Based on what you've just shared, you are currently learning **LangChain** and **CrewAI**, which are both powerful frameworks used in the AI space—LangChain for developing LLM-powered applications and workflows, and CrewAI for orchestrating collaborative multi-agent systems."

## 3) Token Buffer Memory

In [21]:
history = InMemoryChatMessageHistory()

In [22]:
history.messages

[]

In [23]:
history.add_user_message(
    "I am studying Computer Science and currently focusing on "
    "machine learning, deep learning, natural language processing, "
    "generative AI, and AI agents."
)

history.add_ai_message(
    "That's a broad and interesting learning path."
)

history.add_user_message(
    "I am also learning LangChain and I want to understand "
    "how memory works in conversational applications."
)

history.add_ai_message(
    "LangChain provides several approaches for managing conversation history."
)

history.add_user_message(
    "I am currently working on understanding Buffer Memory, "
    "Window Memory, Token Buffer Memory, and Summary Memory."
)

history.add_ai_message(
    "These approaches help control how much conversation context is sent to the model."
)

In [24]:
recent_history = trim_messages(
    history.messages,
    strategy="last",
    max_tokens=100,
    token_counter=model,
)

In [25]:
recent_history

[AIMessage(content="That's a broad and interesting learning path.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I am also learning LangChain and I want to understand how memory works in conversational applications.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='LangChain provides several approaches for managing conversation history.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I am currently working on understanding Buffer Memory, Window Memory, Token Buffer Memory, and Summary Memory.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='These approaches help control how much conversation context is sent to the model.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [26]:
response = chain.invoke({
    "history": recent_history,
    "input": "What am I currently learning about?"
})

print(response)

You are currently learning about **memory in conversational applications using LangChain**, specifically focusing on four types of memory:

1. **Buffer Memory** (storing the entire conversation history)
2. **Window Memory** (keeping only a fixed number of recent interactions)
3. **Token Buffer Memory** (limiting history based on token counts rather than message counts)
4. **Summary Memory** (using an LLM to maintain a running summary of the conversation rather than storing raw text)


## 4) Summary Memory

In [30]:
history = InMemoryChatMessageHistory()

In [31]:
history.messages

[]

In [32]:
history.add_user_message(
    "My name is Bishoy Amgad."
)

history.add_ai_message(
    "Nice to meet you, Bishoy!"
)

history.add_user_message(
    "I study Computer Science at Ain Shams University, "
    "Faculty of Computer and Information Sciences."
)

history.add_ai_message(
    "That's great! Computer Science gives you a strong foundation "
    "for AI and software development."
)

history.add_user_message(
    "I am learning Python, Machine Learning, Deep Learning, NLP, "
    "Generative AI, LangChain, and CrewAI."
)

history.add_ai_message(
    "That's a solid AI learning path. LangChain and CrewAI will "
    "help you build LLM applications and AI agents."
)

In [27]:
summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a conversation summarizer.

Summarize the important facts and context from the conversation.
Keep the summary short and useful for future conversation."""
    ),
    (
        "human",
        """
Previous summary:
{summary}

New conversation:
{conversation}

Create an updated summary.
"""
    )
])

In [28]:
summary_chain = summary_prompt | model | StrOutputParser()

In [29]:
summary = ""

In [37]:
summary = summary_chain.invoke({
    "summary": summary,
    "conversation": history.messages
})

In [38]:
print(summary)

- **User Name:** Bishoy Amgad
- **Education:** Computer Science student at Ain Shams University (Faculty of Computer and Information Sciences)
- **Skills/Interests:** Python, Machine Learning, Deep Learning, NLP, Generative AI, LangChain, and CrewAI


## 5) Vector Memory

In [42]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [43]:
memories = [
    Document(
        page_content="Bishoy is a Computer Science student at Ain Shams University."
    ),
    Document(
        page_content="Bishoy is learning Python and Machine Learning."
    ),
    Document(
        page_content="Bishoy is learning LangChain and Generative AI."
    ),
    Document(
        page_content="Bishoy is learning CrewAI and AI Agents."
    )
]

In [44]:
vectorstore = FAISS.from_documents(
    memories,
    embeddings
)

In [49]:
results = vectorstore.similarity_search(
    "What technologies am I learning?"
)

In [50]:
for result in results:
    print(result.page_content)

Bishoy is learning LangChain and Generative AI.
Bishoy is learning Python and Machine Learning.
Bishoy is learning CrewAI and AI Agents.
Bishoy is a Computer Science student at Ain Shams University.
